# 🔬 Complete Appraisal PDF Detection & Mass Generation System
**Full OCR + CV Analysis → Complete Field Detection → Structured Generation**

In [ ]:
# Install all dependencies
!pip install -q PyMuPDF pdf2image pytesseract opencv-python-headless reportlab Faker pandas pillow numpy tabulate
!apt-get install -qq tesseract-ocr poppler-utils
print("✅ Dependencies ready!")

In [ ]:
# Upload PDF
from google.colab import files
print("📤 Upload your appraisal PDF...")
uploaded = files.upload()
INPUT_PDF = list(uploaded.keys())[0]
print(f"✅ Loaded: {INPUT_PDF}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 1: COMPLETE PDF DETECTION ENGINE
# ═══════════════════════════════════════════════════════════════════
import fitz
import cv2
import numpy as np
from PIL import Image
import pytesseract
from pdf2image import convert_from_path
import json
import re
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict

@dataclass
class DetectedElement:
    """Represents any detected element in PDF"""
    element_type: str  # text, table, image, line, checkbox, signature
    page: int
    x: int
    y: int
    w: int
    h: int
    content: Any = None
    semantic_label: str = ""
    is_dynamic: bool = True  # False = static (don't change)
    confidence: float = 0.0

class CompletePDFDetector:
    """Detects EVERYTHING in the PDF"""
    
    # All sections in a URAR appraisal
    APPRAISAL_SECTIONS = [
        'meta', 'subject_property', 'borrower', 'lender', 'appraiser',
        'assignment', 'site', 'improvements', 'contract', 'neighborhood',
        'sales_comparison', 'cost_approach', 'income_approach',
        'reconciliation', 'certifications', 'addendums', 'photos', 'maps'
    ]
    
    # Static content (never change)
    STATIC_CONTENT = [
        'certifications', 'definitions', 'license_pages',
        'insurance_pages', 'maps', 'photos', 'signatures'
    ]
    
    # Dynamic content patterns
    FIELD_PATTERNS = {
        # Subject Property
        'property_address': r'Property\s*Address[:\s]*([^\n]{5,50})',
        'city': r'City[:\s]*([A-Za-z\s]{2,30})',
        'state': r'State[:\s]*([A-Z]{2})',
        'zip': r'Zip\s*(?:Code)?[:\s]*(\d{5}(?:-\d{4})?)',
        'county': r'County[:\s]*([A-Za-z\s-]{3,40})',
        'legal_description': r'Legal\s*Description[:\s]*([^\n]{10,100})',
        'parcel_number': r'(?:Parcel|APN|Tax\s*ID)[:\s#]*([\d-]{5,20})',
        'tax_year': r'Tax\s*Year[:\s]*(\d{4})',
        'taxes': r'(?:Real\s*Estate\s*)?Taxes[:\s$]*([\d,]+)',
        'occupancy': r'Occupant[:\s]*(Owner|Tenant|Vacant)',
        
        # Borrower/Lender
        'borrower_name': r'Borrower[:\s]*([A-Za-z\s]{3,50})',
        'lender_name': r'Lender(?:\/Client)?[:\s]*([^\n]{5,60})',
        'lender_address': r'Lender.*?Address[:\s]*([^\n]{10,80})',
        
        # Appraiser
        'appraiser_name': r'Appraiser[:\s]*([A-Za-z\s]{3,50})',
        'appraiser_company': r'Company\s*(?:Name)?[:\s]*([^\n]{5,60})',
        'license_number': r'(?:State\s*)?License\s*#?[:\s]*([A-Z]{1,2}\d{3,6})',
        'license_state': r'State[:\s]*([A-Z]{2})(?=.*License)',
        'license_expiry': r'Expir(?:ation|es)[:\s]*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',
        
        # Dates
        'inspection_date': r'(?:Date\s*of\s*)?Inspection[:\s]*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',
        'effective_date': r'Effective\s*Date[:\s]*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',
        'report_date': r'(?:Report|Signature)\s*Date[:\s]*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',
        
        # Site
        'site_area_acres': r'Site[:\s]*([\d.]+)\s*(?:Acres|ac)',
        'site_area_sqft': r'Site[:\s]*([\d,]+)\s*(?:SF|sq\.?\s*ft)',
        'zoning': r'Zoning\s*(?:Classification)?[:\s]*([A-Z0-9-]{1,10})',
        'zoning_description': r'Zoning\s*Description[:\s]*([^\n]{5,50})',
        'flood_zone': r'FEMA\s*(?:Flood)?\s*Zone[:\s]*([A-Z0-9]{1,5})',
        'flood_map': r'FEMA\s*Map\s*#?[:\s]*([\dA-Z]+)',
        
        # Improvements
        'year_built': r'Year\s*Built[:\s]*(\d{4})',
        'effective_age': r'Effective\s*Age[:\s]*(\d{1,3})',
        'design_style': r'Design\s*(?:Style)?[:\s]*([A-Za-z0-9\s;]+)',
        'quality': r'Quality[:\s]*([CQ][1-6])',
        'condition': r'Condition[:\s]*([C][1-6])',
        'stories': r'(?:No\.?\s*of\s*)?Stories[:\s]*([\d.]+)',
        'rooms_total': r'(?:Total\s*)?Rooms[:\s]*(\d{1,2})',
        'bedrooms': r'(?:Bedrooms|BR)[:\s]*(\d{1,2})',
        'bathrooms': r'(?:Bathrooms|Bath|BA)[:\s]*([\d.]+)',
        'gla': r'(?:Gross\s*Living\s*Area|GLA)[:\s]*([\d,]+)',
        'basement': r'Basement[:\s]*([A-Za-z\s]+?)(?=\s*(?:Finished|None|$))',
        'foundation': r'Foundation[:\s]*([A-Za-z\s]+)',
        'roof': r'Roof\s*(?:Surface)?[:\s]*([A-Za-z\s]+)',
        'heating': r'Heating[:\s]*([A-Za-z\s]+)',
        'cooling': r'Cooling[:\s]*([A-Za-z\s]+)',
        'garage': r'Garage[:\s]*([A-Za-z0-9\s]+)',
        
        # Contract
        'contract_price': r'Contract\s*(?:Sale)?\s*Price[:\s$]*([\d,]+)',
        'contract_date': r'Date\s*of\s*(?:Contract|Sale)[:\s]*(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',
        
        # Values
        'appraised_value': r'(?:Appraised|Opinion\s*of|Market)\s*Value[:\s$]*([\d,]+)',
        'cost_site_value': r'Site\s*Value[:\s$]*([\d,]+)',
        'cost_replacement': r'(?:Replacement|Reproduction)\s*Cost[:\s$]*([\d,]+)',
        'cost_depreciation': r'(?:Less\s*)?Depreciation[:\s$]*([\d,]+)',
        'cost_indicated': r'(?:Indicated\s*Value.*?Cost)[:\s$]*([\d,]+)',
        
        # Comparables (will need special handling)
        'comp_address': r'(?:Comparable|Comp)\s*\d[:\s]*([^\n]{10,50})',
        'comp_sale_price': r'Sale\s*Price[:\s$]*([\d,]+)',
        'comp_sale_date': r'Date\s*(?:of\s*)?Sale[:\s]*(\d{1,2}[/-]\d{2,4})',
    }
    
    def __init__(self, pdf_path, dpi=150):
        self.pdf_path = pdf_path
        self.dpi = dpi
        self.doc = fitz.open(pdf_path)
        self.page_images = []
        self.elements: List[DetectedElement] = []
        self.master_data = {}
        self.page_layouts = []
        
    def detect_all(self):
        """Run complete detection pipeline"""
        print(f"\n{'='*60}")
        print(f"🔬 COMPLETE PDF DETECTION - {len(self.doc)} pages")
        print(f"{'='*60}\n")
        
        self._render_pages()
        self._detect_text_blocks()
        self._detect_tables()
        self._detect_images()
        self._detect_lines_borders()
        self._detect_checkboxes()
        self._classify_sections()
        self._extract_all_fields()
        self._identify_static_dynamic()
        
        self._print_detection_report()
        return self
    
    def _render_pages(self):
        print("📸 Rendering pages to images...")
        self.page_images = convert_from_path(self.pdf_path, dpi=self.dpi)
        for i, img in enumerate(self.page_images):
            self.page_layouts.append({
                'page': i + 1,
                'width': img.width,
                'height': img.height,
                'dpi': self.dpi
            })
        print(f"   ✅ Rendered {len(self.page_images)} pages")
    
    def _detect_text_blocks(self):
        print("📝 Detecting text blocks with OCR...")
        total_blocks = 0
        for i, img in enumerate(self.page_images):
            gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
            ocr = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT)
            
            for j in range(len(ocr['text'])):
                text = ocr['text'][j].strip()
                conf = int(ocr['conf'][j])
                if text and conf > 30:
                    self.elements.append(DetectedElement(
                        element_type='text',
                        page=i + 1,
                        x=ocr['left'][j],
                        y=ocr['top'][j],
                        w=ocr['width'][j],
                        h=ocr['height'][j],
                        content=text,
                        confidence=conf
                    ))
                    total_blocks += 1
        print(f"   ✅ Found {total_blocks} text blocks")
    
    def _detect_tables(self):
        print("📊 Detecting tables...")
        table_count = 0
        for i, img in enumerate(self.page_images):
            gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
            thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
            
            # Detect horizontal and vertical lines
            h_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
            v_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 40))
            h_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, h_kernel)
            v_lines = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, v_kernel)
            
            # Combine to find table regions
            table_mask = cv2.add(h_lines, v_lines)
            contours, _ = cv2.findContours(table_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for cnt in contours:
                x, y, w, h = cv2.boundingRect(cnt)
                if w > 200 and h > 50:
                    self.elements.append(DetectedElement(
                        element_type='table',
                        page=i + 1,
                        x=x, y=y, w=w, h=h,
                        content={'rows': 0, 'cols': 0}
                    ))
                    table_count += 1
        print(f"   ✅ Found {table_count} table regions")
    
    def _detect_images(self):
        print("🖼️ Detecting embedded images...")
        img_count = 0
        for i, page in enumerate(self.doc):
            for img_info in page.get_images():
                xref = img_info[0]
                self.elements.append(DetectedElement(
                    element_type='image',
                    page=i + 1,
                    x=0, y=0, w=0, h=0,
                    content={'xref': xref},
                    is_dynamic=False  # Images are static
                ))
                img_count += 1
        print(f"   ✅ Found {img_count} images")
    
    def _detect_lines_borders(self):
        print("📏 Detecting lines and borders...")
        line_count = 0
        for i, img in enumerate(self.page_images):
            gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
            edges = cv2.Canny(gray, 50, 150)
            lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)
            if lines is not None:
                for line in lines:
                    x1, y1, x2, y2 = line[0]
                    self.elements.append(DetectedElement(
                        element_type='line',
                        page=i + 1,
                        x=x1, y=y1, w=x2-x1, h=y2-y1,
                        is_dynamic=False
                    ))
                    line_count += 1
        print(f"   ✅ Found {line_count} lines/borders")
    
    def _detect_checkboxes(self):
        print("☑️ Detecting checkboxes...")
        cb_count = 0
        for i, img in enumerate(self.page_images):
            gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
            thresh = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY_INV)[1]
            contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
            
            for cnt in contours:
                x, y, w, h = cv2.boundingRect(cnt)
                aspect = w / h if h > 0 else 0
                if 10 < w < 30 and 10 < h < 30 and 0.8 < aspect < 1.2:
                    self.elements.append(DetectedElement(
                        element_type='checkbox',
                        page=i + 1,
                        x=x, y=y, w=w, h=h
                    ))
                    cb_count += 1
        print(f"   ✅ Found {cb_count} checkboxes")
    
    def _classify_sections(self):
        print("🏷️ Classifying sections...")
        section_keywords = {
            'subject_property': ['property', 'address', 'subject'],
            'borrower': ['borrower', 'owner'],
            'lender': ['lender', 'client', 'mortgage'],
            'appraiser': ['appraiser', 'license', 'certification'],
            'site': ['site', 'lot', 'zoning', 'flood'],
            'improvements': ['improvement', 'room', 'bedroom', 'bath', 'gla'],
            'sales_comparison': ['comparable', 'sale', 'adjustment'],
            'cost_approach': ['cost', 'replacement', 'depreciation'],
            'reconciliation': ['reconciliation', 'final', 'opinion'],
            'neighborhood': ['neighborhood', 'market', 'location'],
            'photos': ['photo', 'picture', 'front', 'rear'],
            'maps': ['map', 'location', 'plat']
        }
        
        for elem in self.elements:
            if elem.element_type == 'text' and elem.content:
                text_lower = elem.content.lower()
                for section, keywords in section_keywords.items():
                    if any(kw in text_lower for kw in keywords):
                        elem.semantic_label = section
                        break
    
    def _extract_all_fields(self):
        print("📋 Extracting all fields...")
        all_text = " ".join([e.content for e in self.elements if e.element_type == 'text' and e.content])
        
        for field, pattern in self.FIELD_PATTERNS.items():
            match = re.search(pattern, all_text, re.IGNORECASE | re.DOTALL)
            if match:
                self.master_data[field] = match.group(1).strip()
        
        print(f"   ✅ Extracted {len(self.master_data)} fields")
    
    def _identify_static_dynamic(self):
        print("🔄 Identifying static vs dynamic content...")
        static_keywords = ['certification', 'definition', 'license', 'insurance', 'uspap', 'copyright']
        
        for elem in self.elements:
            if elem.element_type in ['image', 'line']:
                elem.is_dynamic = False
            elif elem.element_type == 'text' and elem.content:
                if any(kw in elem.content.lower() for kw in static_keywords):
                    elem.is_dynamic = False
    
    def _print_detection_report(self):
        print(f"\n{'='*60}")
        print("📊 DETECTION REPORT")
        print(f"{'='*60}\n")
        
        # Count by type
        counts = defaultdict(int)
        for e in self.elements:
            counts[e.element_type] += 1
        
        print("Element Counts:")
        for etype, count in counts.items():
            print(f"   {etype:15} : {count}")
        
        print(f"\n📝 Extracted Fields ({len(self.master_data)}):")
        for k, v in list(self.master_data.items())[:15]:
            print(f"   {k:25} : {v[:40] if len(str(v)) > 40 else v}")
        if len(self.master_data) > 15:
            print(f"   ... and {len(self.master_data) - 15} more fields")

# Run complete detection
detector = CompletePDFDetector(INPUT_PDF)
detector.detect_all()

In [ ]:
# View all detected data
print("\n📋 COMPLETE MASTER DATA MODEL:")
print(json.dumps(detector.master_data, indent=2))

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 2: INTELLIGENT DATA GENERATION ENGINE
# ═══════════════════════════════════════════════════════════════════
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()

class IntelligentDataGenerator:
    """Generates realistic, consistent appraisal data"""
    
    STATES = ['FL', 'CA', 'TX', 'NY', 'AZ', 'CO', 'GA', 'NC', 'VA', 'WA', 'OR', 'NV', 'TN', 'SC', 'PA']
    QUALITIES = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6']
    CONDITIONS = ['C1', 'C2', 'C3', 'C4', 'C5', 'C6']
    FLOOD_ZONES = ['X', 'A', 'AE', 'AH', 'AO', 'VE']
    ZONING = ['R-1', 'R-2', 'R-3', 'R-4', 'PUD', 'A-1', 'AU', 'RM', 'RS']
    DESIGNS = ['Colonial', 'Ranch', 'Contemporary', 'Split Level', 'Cape Cod', 'Tudor', 'Mediterranean']
    
    def __init__(self):
        self.used_addresses = set()
        self.used_parcels = set()
        self.generated_count = 0
    
    def generate(self) -> Dict:
        """Generate one complete, consistent appraisal dataset"""
        self.generated_count += 1
        
        # Core property characteristics (these drive other values)
        state = random.choice(self.STATES)
        year_built = random.randint(1950, 2023)
        gla = random.randint(1200, 5500)
        site_acres = round(random.uniform(0.15, 8.0), 3)
        quality = random.choice(self.QUALITIES[:4])
        condition = random.choice(self.CONDITIONS[:4])
        
        # Value calculation (correlated with characteristics)
        q_mult = {'Q1': 1.4, 'Q2': 1.2, 'Q3': 1.0, 'Q4': 0.85}[quality]
        c_mult = {'C1': 1.15, 'C2': 1.05, 'C3': 0.95, 'C4': 0.85}[condition]
        base_value = gla * random.randint(180, 450) * q_mult * c_mult
        appraised_value = int(round(base_value, -3))  # Round to nearest 1000
        
        # Dates (logically consistent)
        inspection_date = fake.date_between(start_date='-30d', end_date='today')
        effective_date = inspection_date
        report_date = inspection_date + timedelta(days=random.randint(3, 14))
        contract_date = inspection_date - timedelta(days=random.randint(10, 60))
        
        # Generate unique address
        address = self._unique_address()
        parcel = self._unique_parcel()
        
        # Room configuration based on GLA
        bedrooms = max(2, min(6, gla // 600))
        bathrooms = round(bedrooms * 0.75 + random.uniform(0, 1), 1)
        total_rooms = bedrooms + random.randint(3, 6)
        
        # Cost approach (must reconcile)
        site_value = int(appraised_value * random.uniform(0.2, 0.4))
        replacement_cost = gla * random.randint(150, 350)
        depreciation = int(replacement_cost * (datetime.now().year - year_built) * 0.01)
        cost_indicated = site_value + replacement_cost - depreciation
        
        # Generate comparables
        comparables = self._generate_comparables(appraised_value, gla, site_acres)
        
        return {
            # Meta
            'file_no': f"{random.randint(2300, 2500)}-{random.randint(10000, 99999)}",
            'fha_case_no': str(random.randint(1000000, 9999999)),
            
            # Subject Property
            'property_address': address,
            'city': fake.city(),
            'state': state,
            'zip': fake.zipcode(),
            'county': fake.city() + ' County',
            'legal_description': f"LOT {random.randint(1,200)}, BLK {random.randint(1,50)}, {fake.street_name()} SUBDIVISION",
            'parcel_number': parcel,
            'tax_year': '2023',
            'taxes': str(int(appraised_value * 0.012)),
            'occupancy': random.choice(['Owner', 'Tenant', 'Vacant']),
            
            # Borrower/Lender
            'borrower_name': fake.name(),
            'lender_name': fake.company() + random.choice([' Mortgage', ' Lending', ' Bank', ' Credit Union']),
            'lender_address': fake.address().replace('\n', ', '),
            
            # Appraiser
            'appraiser_name': fake.name(),
            'appraiser_company': fake.company() + random.choice([' Appraisals', ' Valuations', ' Appraisal Services']),
            'license_number': f"R{random.choice(['D','G','A'])}{random.randint(1000, 9999)}",
            'license_state': state,
            'license_expiry': fake.date_between(start_date='+1y', end_date='+3y').strftime('%m/%d/%Y'),
            
            # Dates
            'inspection_date': inspection_date.strftime('%m/%d/%Y'),
            'effective_date': effective_date.strftime('%m/%d/%Y'),
            'report_date': report_date.strftime('%m/%d/%Y'),
            
            # Site
            'site_area_acres': str(site_acres),
            'site_area_sqft': str(int(site_acres * 43560)),
            'zoning': random.choice(self.ZONING),
            'zoning_description': random.choice(['Single Family', 'Residential', 'Agriculture', 'PUD']),
            'flood_zone': random.choice(self.FLOOD_ZONES),
            'flood_map': f"12{random.randint(10,99)}C{random.randint(100,999)}L",
            
            # Improvements
            'year_built': str(year_built),
            'effective_age': str(datetime.now().year - year_built),
            'design_style': random.choice(self.DESIGNS),
            'quality': quality,
            'condition': condition,
            'stories': str(random.choice([1, 1.5, 2, 2.5])),
            'rooms_total': str(total_rooms),
            'bedrooms': str(bedrooms),
            'bathrooms': str(bathrooms),
            'gla': f"{gla:,}",
            'basement': random.choice(['Full', 'Partial', 'Crawl', 'Slab', 'None']),
            'foundation': random.choice(['Concrete Slab', 'Basement', 'Crawl Space', 'Pier']),
            'roof': random.choice(['Shingle', 'Tile', 'Metal', 'Slate', 'Flat']),
            'heating': random.choice(['Central', 'Forced Air', 'Heat Pump', 'Radiant']),
            'cooling': random.choice(['Central', 'Window', 'None', 'High Efficiency']),
            'garage': random.choice(['1 Car', '2 Car', '3 Car', 'Carport', 'None']),
            
            # Contract
            'contract_price': f"{appraised_value + random.randint(-50000, 100000):,}",
            'contract_date': contract_date.strftime('%m/%d/%Y'),
            
            # Values
            'appraised_value': f"{appraised_value:,}",
            'cost_site_value': f"{site_value:,}",
            'cost_replacement': f"{replacement_cost:,}",
            'cost_depreciation': f"{depreciation:,}",
            'cost_indicated': f"{cost_indicated:,}",
            
            # Comparables
            'comparables': comparables,
            
            # Narrative
            'neighborhood_desc': fake.paragraph(nb_sentences=3),
            'reconciliation_summary': fake.paragraph(nb_sentences=4),
            'market_conditions': random.choice(['Stable', 'Increasing', 'Declining'])
        }
    
    def _unique_address(self):
        while True:
            addr = f"{random.randint(100, 19999)} {fake.street_name()} {random.choice(['St', 'Ave', 'Blvd', 'Dr', 'Ln', 'Ct'])}"
            if addr not in self.used_addresses:
                self.used_addresses.add(addr)
                return addr
    
    def _unique_parcel(self):
        while True:
            parcel = f"{random.randint(10,99)}-{random.randint(1000,9999)}-{random.randint(100,999)}-{random.randint(1000,9999)}"
            if parcel not in self.used_parcels:
                self.used_parcels.add(parcel)
                return parcel
    
    def _generate_comparables(self, subject_value, subject_gla, subject_site):
        comps = []
        for i in range(random.randint(3, 6)):
            comp_gla = subject_gla + random.randint(-500, 800)
            comp_site = subject_site + random.uniform(-1, 2)
            comp_price = subject_value + random.randint(-150000, 250000)
            
            # Calculate realistic adjustments
            gla_adj = (subject_gla - comp_gla) * random.randint(80, 150)
            site_adj = (subject_site - comp_site) * random.randint(5000, 20000)
            total_adj = int(gla_adj + site_adj + random.randint(-10000, 10000))
            
            comps.append({
                'id': i + 1,
                'address': f"{random.randint(100, 9999)} {fake.street_name()} {random.choice(['St', 'Ave', 'Dr'])}",
                'city': fake.city(),
                'distance': round(random.uniform(0.3, 5.0), 1),
                'sale_price': comp_price,
                'sale_date': fake.date_between(start_date='-8m', end_date='-1m').strftime('%m/%Y'),
                'gla': comp_gla,
                'site': round(comp_site, 2),
                'adjustment': total_adj,
                'adjusted_price': comp_price + total_adj
            })
        return comps

# Test the generator
gen = IntelligentDataGenerator()
sample = gen.generate()
print("✅ Sample Generated:")
print(f"   📍 {sample['property_address']}, {sample['city']}, {sample['state']} {sample['zip']}")
print(f"   💰 Appraised: ${sample['appraised_value']}")
print(f"   🏠 {sample['bedrooms']} BR / {sample['bathrooms']} BA, {sample['gla']} SF, Built {sample['year_built']}")
print(f"   👤 Borrower: {sample['borrower_name']}")
print(f"   📊 {len(sample['comparables'])} Comparables")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 3: PDF REGENERATION (OVERLAY METHOD)
# ═══════════════════════════════════════════════════════════════════
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.colors import white, black, Color

class PDFRegenerator:
    def __init__(self, detector: CompletePDFDetector):
        self.detector = detector
        self.original_data = detector.master_data
    
    def regenerate(self, output_path: str, new_data: Dict):
        """Create PDF with original background + new data overlay"""
        c = canvas.Canvas(output_path, pagesize=letter)
        page_w, page_h = letter
        
        for i, page_img in enumerate(self.detector.page_images):
            # Save temp image
            img_path = f'/tmp/page_{i}.png'
            page_img.save(img_path, 'PNG')
            
            # Draw original as background
            c.drawImage(img_path, 0, 0, width=page_w, height=page_h)
            
            # Calculate scale
            if i < len(self.detector.page_layouts):
                layout = self.detector.page_layouts[i]
                scale_x = page_w / layout['width']
                scale_y = page_h / layout['height']
                
                # Find text elements on this page that need replacement
                for elem in self.detector.elements:
                    if elem.page == i + 1 and elem.element_type == 'text' and elem.is_dynamic:
                        replacement = self._find_replacement(elem.content, new_data)
                        if replacement:
                            # PDF y-coordinate is from bottom
                            x = elem.x * scale_x
                            y = page_h - (elem.y * scale_y) - (elem.h * scale_y)
                            w = elem.w * scale_x
                            h = elem.h * scale_y
                            
                            # White out and redraw
                            c.setFillColor(white)
                            c.rect(x - 1, y - 2, w + 4, h + 4, fill=True, stroke=False)
                            c.setFillColor(black)
                            font_size = min(max(h * 0.7, 6), 11)
                            c.setFont("Helvetica", font_size)
                            c.drawString(x, y + 2, str(replacement)[:60])
            
            c.showPage()
        
        c.save()
        return output_path
    
    def _find_replacement(self, original_text: str, new_data: Dict):
        """Find if this text should be replaced with new data"""
        if not original_text:
            return None
        
        orig_lower = original_text.lower().strip()
        
        # Check against extracted original values
        for field, orig_value in self.original_data.items():
            if orig_value and field in new_data:
                orig_val_lower = str(orig_value).lower().strip()
                if orig_val_lower == orig_lower or orig_val_lower in orig_lower:
                    return str(new_data[field])
        
        return None

print("✅ Regenerator ready!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STAGE 4: MASS GENERATION
# ═══════════════════════════════════════════════════════════════════
import os
import shutil

NUM_PDFS = 15  # ← Change this to generate more/fewer
OUTPUT_DIR = "generated_appraisals"

# Clean and create output directory
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

# Initialize
data_gen = IntelligentDataGenerator()
regenerator = PDFRegenerator(detector)
generated_files = []
all_data = []

print(f"\n{'='*60}")
print(f"🚀 GENERATING {NUM_PDFS} UNIQUE APPRAISAL PDFs")
print(f"{'='*60}\n")

for i in range(1, NUM_PDFS + 1):
    # Generate unique data
    data = data_gen.generate()
    all_data.append(data)
    
    # Create PDF
    filename = f"synthetic_appraisal_{i:03d}.pdf"
    path = os.path.join(OUTPUT_DIR, filename)
    regenerator.regenerate(path, data)
    generated_files.append(path)
    
    print(f"✅ [{i:2d}/{NUM_PDFS}] {filename}")
    print(f"   📍 {data['property_address']}, {data['city']}, {data['state']}")
    print(f"   💰 ${data['appraised_value']} | {data['gla']} SF | {data['bedrooms']} BR")
    print(f"   👤 {data['borrower_name']}\n")

print(f"{'='*60}")
print(f"🎉 Generated {len(generated_files)} unique PDFs!")
print(f"📁 Location: {OUTPUT_DIR}/")

In [ ]:
# Verify uniqueness
import hashlib

print("\n🔍 Verifying uniqueness...")
hashes = {}
unique = True
for f in generated_files:
    with open(f, 'rb') as file:
        h = hashlib.md5(file.read()).hexdigest()
        if h in hashes:
            print(f"❌ Duplicate: {f}")
            unique = False
        hashes[h] = f

if unique:
    print(f"✅ All {len(generated_files)} PDFs are unique!")

In [ ]:
# Create summary CSV
import pandas as pd

summary = []
for i, d in enumerate(all_data):
    summary.append({
        'File': f'synthetic_appraisal_{i+1:03d}.pdf',
        'Borrower': d['borrower_name'],
        'Address': d['property_address'],
        'City': d['city'],
        'State': d['state'],
        'Zip': d['zip'],
        'Appraised Value': f"${d['appraised_value']}",
        'GLA': d['gla'],
        'BR/BA': f"{d['bedrooms']}/{d['bathrooms']}",
        'Year Built': d['year_built'],
        'Parcel': d['parcel_number']
    })

df = pd.DataFrame(summary)
df.to_csv('appraisal_data_summary.csv', index=False)
print("📊 Summary:")
display(df)

In [ ]:
# Download everything
import zipfile

zip_name = "appraisal_pdfs.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in generated_files:
        zf.write(f, os.path.basename(f))
    zf.write('appraisal_data_summary.csv')

print(f"\n📦 Created {zip_name}")
print(f"   Size: {os.path.getsize(zip_name) / (1024*1024):.2f} MB")
print(f"   Contains: {len(generated_files)} PDFs + summary CSV")

files.download(zip_name)
print("\n⬇️ Download started!")